# Deploy Decision 2.0 Sol 2B on SageMaker AI

Deploy [Decision 2.0 Sol 2B](https://huggingface.co/vllm-sr/Decision-2.0-Sol-2B)
on one real-time endpoint (`ml.g6.xlarge`, falling back to `ml.g5.xlarge`). Invoke its native Choice, Yes/No
(`noul`), and Score outputs through Transformers `AutoModel` and `system_one`.
The checkpoint and its remote runtime code are pinned to the same revision.

Read [README.md](README.md) for prerequisites, licensing, and runtime details.
It runs on the maintained AWS PyTorch 2.14 SageMaker DLC, through a small serving
image that this notebook builds with AWS CodeBuild. This is an evaluation example.

Run from this directory. Run the cleanup cell when finished, including after a
failed deployment once SageMaker allows endpoint deletion.

## 1. Environment preparation

In [ ]:
%pip install --upgrade 'boto3>=1.43.1,<2' --quiet

In [ ]:
import json
import math
import os
import statistics
import tarfile
import time
import uuid
from pathlib import Path

import boto3
from botocore.config import Config
from botocore.exceptions import ClientError

session = boto3.Session()
region = session.region_name or "us-west-2"
sm = session.client("sagemaker", region_name=region)
runtime = session.client(
    "sagemaker-runtime", region_name=region,
    config=Config(read_timeout=70, retries={"max_attempts": 0}),
)
s3 = session.client("s3", region_name=region)
iam = session.client("iam")
logs = session.client("logs", region_name=region)
identity = session.client("sts").get_caller_identity()
account_id = identity["Account"]

# Set this when the notebook's own role is not a SageMaker execution role.
execution_role = os.environ.get("SAGEMAKER_EXECUTION_ROLE_ARN")
bucket = os.environ.get("SAGEMAKER_ARTIFACT_BUCKET", f"sagemaker-{region}-{account_id}")
# Instance types in priority order. With more than one, the endpoint uses a capacity-aware
# instance pool and falls back to the next type when the first has no capacity.
instance_types = os.environ.get("SAGEMAKER_INSTANCE_TYPES", "ml.g6.xlarge,ml.g5.xlarge").split(",")
# Set this to reuse a serving image you already built; otherwise section 2 builds one.
inference_image = os.environ.get("SAGEMAKER_INFERENCE_IMAGE_URI")
codebuild_role_arn = os.environ.get("CODEBUILD_ROLE_ARN")
# The serving image runs PyTorch 2.14 with CUDA 13. The default SageMaker AMIs for ml.g5 and
# ml.g4dn have NVIDIA driver 470 (CUDA 11.4) and cannot start it, so use the driver 580 AMI.
inference_ami_version = os.environ.get("SAGEMAKER_INFERENCE_AMI_VERSION", "al2023-ami-sagemaker-inference-gpu-4-1")
name = f"decision-sol-2b-{uuid.uuid4().hex[:12]}"
artifact_key = f"decision-sol/{name}/model.tar.gz"
artifact_version = None
example_dir = Path.cwd()
archive_path = example_dir / f"{name}.tar.gz"
for filename in ["inference.py", "requirements.txt", "example-request.json"]:
    if not (example_dir / filename).is_file():
        raise FileNotFoundError(f"Run the notebook from its example directory: missing {filename}")
print(f"Region: {region}; instances: {', '.join(instance_types)}; AMI: {inference_ami_version}; endpoint: {name}")

### Resolve the execution role

In Studio the current role may already be appropriate. Outside Studio,
set `SAGEMAKER_EXECUTION_ROLE_ARN` to an existing SageMaker role. This cell
reads the role's actual ARN through IAM, preserving paths such as
`service-role/`, and checks its SageMaker trust.

In [ ]:
if execution_role is None:
    caller_arn = identity["Arn"]
    if ":assumed-role/" not in caller_arn:
        raise ValueError("Set SAGEMAKER_EXECUTION_ROLE_ARN to an existing SageMaker execution role.")
    role_name = caller_arn.split(":assumed-role/", 1)[1].split("/", 1)[0]
else:
    role_name = execution_role.rsplit("/", 1)[-1]
role_info = iam.get_role(RoleName=role_name)["Role"]
if execution_role is not None and execution_role != role_info["Arn"]:
    raise ValueError("The configured execution-role ARN does not match the resolved IAM role.")
execution_role = role_info["Arn"]
trusts_sagemaker = any(
    statement.get("Effect") == "Allow"
    and "sagemaker.amazonaws.com" in (
        [statement.get("Principal", {}).get("Service")]
        if isinstance(statement.get("Principal", {}).get("Service"), str)
        else statement.get("Principal", {}).get("Service", [])
    )
    for statement in role_info["AssumeRolePolicyDocument"]["Statement"]
)
if not trusts_sagemaker:
    raise ValueError(
        "The current role does not trust SageMaker. Set SAGEMAKER_EXECUTION_ROLE_ARN "
        "to an appropriate existing execution role and rerun setup."
    )
print("Execution role resolved and SageMaker trust verified.")

## 2. Build the serving image

There is no AWS PyTorch inference DLC after 2.6, and that image is past its end of
support. This example runs on the maintained PyTorch 2.14 SageMaker DLC through the small
shared image in [03-features/decision-model-serving-image](../../../03-features/decision-model-serving-image/).
It adds `serve`, which answers `/ping` and `/invocations` by calling this example's
`model_fn` and `transform_fn`.

AWS CodeBuild builds the image and pushes it to a private Amazon ECR repository in your
account, in about 7 minutes. Set `CODEBUILD_ROLE_ARN` to a CodeBuild service role that can
read the artifact bucket, write build logs, and push to Amazon ECR. To reuse an image you
already built, set `SAGEMAKER_INFERENCE_IMAGE_URI` and this cell skips the build.

In [ ]:
import sys

if inference_image is None:
    if not codebuild_role_arn:
        raise ValueError("Set CODEBUILD_ROLE_ARN, or SAGEMAKER_INFERENCE_IMAGE_URI to reuse an image.")
    image_dir = example_dir.parents[2] / "03-features" / "decision-model-serving-image"
    if not image_dir.is_dir():
        raise FileNotFoundError(f"{image_dir} not found. Run this notebook from a clone of the repository.")
    sys.path.insert(0, str(image_dir))
    from build_image import build_image

    inference_image = build_image(region=region, bucket=bucket, codebuild_role_arn=codebuild_role_arn)
print(f"Serving image: {inference_image}")

## 3. Prepare the model artifact

Package the companion handler and requirements under `code/`, as expected by
the serving image. The worker loads the pinned Hugging Face
checkpoint and its bundled runtime using `trust_remote_code=True`. Review the
published code before adopting it for another workload.

The default artifact bucket is shared and is preserved during cleanup. Capture
the uploaded object's version directly from the upload response.

In [ ]:
try:
    s3.head_bucket(Bucket=bucket)
except ClientError as error:
    if error.response["Error"]["Code"] not in {"404", "NoSuchBucket", "NotFound"}:
        raise
    create_args = {"Bucket": bucket}
    if region != "us-east-1":
        create_args["CreateBucketConfiguration"] = {"LocationConstraint": region}
    s3.create_bucket(**create_args)
    s3.put_public_access_block(
        Bucket=bucket,
        PublicAccessBlockConfiguration={
            "BlockPublicAcls": True, "IgnorePublicAcls": True,
            "BlockPublicPolicy": True, "RestrictPublicBuckets": True,
        },
    )

with tarfile.open(archive_path, "w:gz") as archive:
    for filename in ["inference.py", "requirements.txt"]:
        archive.add(example_dir / filename, arcname=f"code/{filename}")
with archive_path.open("rb") as artifact:
    uploaded = s3.put_object(Bucket=bucket, Key=artifact_key, Body=artifact)
artifact_version = uploaded.get("VersionId")
model_data_url = f"s3://{bucket}/{artifact_key}"
print("Inference artifact uploaded.")

## 4. Create the model, endpoint configuration, and endpoint

SageMaker allows 1,800 seconds for the container to pass its health check, which
covers loading the model. The serving image runs one process and the adapter serializes
inference, to avoid duplicate weights and overlapping requests.
The model's native runtime retains its own mixed precision: eligible BF16
backbone Linear weights and an FP32 head. Do not override its dtype.

This example omits `VolumeSizeInGB` for the instance's local-storage configuration.

With more than one instance type, the variant uses a capacity-aware instance pool:
SageMaker AI tries the types in priority order. `InferenceAmiVersion` selects the NVIDIA
driver 580 AMI that the CUDA 13 serving image needs.

In [ ]:
sm.create_model(
    ModelName=name,
    ExecutionRoleArn=execution_role,
    PrimaryContainer={
        "Image": inference_image,
        "ModelDataUrl": model_data_url,
        "Environment": {
            "SAGEMAKER_PROGRAM": "inference.py",
            "SAGEMAKER_SUBMIT_DIRECTORY": "/opt/ml/model/code",
            "HF_HOME": "/tmp/huggingface",
            "HF_HUB_DISABLE_TELEMETRY": "1",
            "TOKENIZERS_PARALLELISM": "false",
        },
    },
)
instance_choice = (
    {"InstanceType": instance_types[0]} if len(instance_types) == 1 else {
        "InstancePools": [{"InstanceType": t, "Priority": p} for p, t in enumerate(instance_types, 1)],
        "VariantInstanceProvisionTimeoutInSeconds": 1800,
    }
)
sm.create_endpoint_config(
    EndpointConfigName=name,
    ProductionVariants=[{
        "VariantName": "AllTraffic", "ModelName": name,
        "InitialInstanceCount": 1, "InitialVariantWeight": 1.0,
        "ModelDataDownloadTimeoutInSeconds": 900,
        "ContainerStartupHealthCheckTimeoutInSeconds": 1800,
        "InferenceAmiVersion": inference_ami_version,
        **instance_choice,
    }],
)
sm.create_endpoint(EndpointName=name, EndpointConfigName=name)
print(f"Endpoint {name} is deploying.")

In [ ]:
sm.get_waiter("endpoint_in_service").wait(
    EndpointName=name, WaiterConfig={"Delay": 30, "MaxAttempts": 120}
)
endpoint = sm.describe_endpoint(EndpointName=name)
assert endpoint["EndpointStatus"] == "InService"
assert endpoint["EndpointConfigName"] == name
assert sum(v["CurrentInstanceCount"] for v in endpoint["ProductionVariants"]) == 1
print("Endpoint is InService with one instance.")

## 5. Verify the runtime and invoke typed questions

The adapter accepts `{"state": ..., "questions": {...}}` through SageMaker's
IAM-authenticated API and calls `model.system_one`. Its diagnostics operation
reports effective versions, device, checkpoint metadata, and memory use.

In [ ]:
def invoke(payload):
    response = runtime.invoke_endpoint(
        EndpointName=name, ContentType="application/json",
        Accept="application/json", Body=json.dumps(payload),
    )
    return json.loads(response["Body"].read())

diagnostics = invoke({"operation": "diagnostics"})
assert diagnostics["device"] == "cuda:0"
assert diagnostics["model_revision"] == "64235bef55dad29387dd16da7c90e038bf2f0972"
assert diagnostics["model"] == "Decision-2.0-Sol-2B"
assert diagnostics["loaded_parameters"] == 1883930944
assert diagnostics["max_input_tokens"] == 16384
assert diagnostics["versions"]["torch"].startswith("2.14.")
assert diagnostics["versions"]["transformers"] == "5.17.0"
assert diagnostics["gpu"].startswith("NVIDIA")
print(json.dumps(diagnostics, indent=2))

### Tool-call readiness: Yes/No, Choice, and Score

The user requested weather without supplying a city, but a proposed tool call
guessed Seattle. Ask whether its arguments are grounded, what should happen
next, and how ready the call is. The harness owns authorization and execution.

In [ ]:
request = json.loads((example_dir / "example-request.json").read_text())
result = invoke(request)
print(json.dumps(result, indent=2))

In [ ]:
assert result["model"] == diagnostics["model"]
answers = result["answers"]
assert set(answers) == {"arguments_grounded", "next_action", "readiness"}
assert answers["arguments_grounded"]["type"] == "noul"
assert math.isfinite(answers["arguments_grounded"]["noul"])
assert 0 <= answers["arguments_grounded"]["noul"] <= 1
for key, kind in [("next_action", "choice"), ("readiness", "score")]:
    answer = answers[key]
    assert answer["type"] == kind
    probabilities = list(answer["probabilities"].values())
    assert all(math.isfinite(p) and 0 <= p <= 1 for p in probabilities)
    assert abs(sum(probabilities) - 1) <= 0.001
assert answers["next_action"]["choice"] in request["questions"]["next_action"]["criteria"]
assert math.isfinite(answers["readiness"]["score"])
assert 0 <= answers["readiness"]["score"] <= 2
print("All three answer types verified.")

### Basic expected-answer smoke checks

These simple cases check the serving path. They are not a representative
benchmark of decision quality or calibration.

In [ ]:
language = invoke({
    "state": "This is a clear sentence written in English.",
    "questions": {"language": {
        "type": "choice", "instructions": "Which language is this sentence written in?",
        "criteria": {"English": "English", "French": "French", "German": "German"},
    }},
})
assert language["answers"]["language"]["choice"] == "English"

city_question = {"city_supplied": {
    "type": "noul", "instructions": "Did the user explicitly provide a city?",
}}
explicit_city = invoke({
    "state": "The user says: What is the current weather in Paris?",
    "questions": city_question,
})
missing_city = invoke({
    "state": "The user says: What is the current weather?",
    "questions": city_question,
})
assert explicit_city["answers"]["city_supplied"]["noul"] >= 0.5
assert missing_city["answers"]["city_supplied"]["noul"] < 0.5
print("Basic expected-answer checks: 3/3.")

### Repeatability and server latency

Three repeats give a hosting smoke measurement, not a production latency
estimate. `latency_ms` measures server-side inference; SDK/network time is
measured separately in the following scaling check.

In [ ]:
repeats = [invoke(request) for _ in range(3)]
assert all(item["answers"] == result["answers"] for item in repeats)
print(f"Median warm server latency: {statistics.median(item['latency_ms'] for item in repeats):.2f} ms")

### Reject oversized state instead of silently truncating it

The oversized request has valid schema and exceeds the checkpoint's
16,384-token budget. The adapter turns native context-window errors into HTTP
422; SageMaker wraps this in `ModelError` and may omit the error body.

In [ ]:
oversized_request = {
    "state": "weather city location " * 10000,
    "questions": city_question,
}
try:
    invoke(oversized_request)
except runtime.exceptions.ModelError as error:
    assert error.response.get("OriginalStatusCode") == 422, error.response
    print("Oversized state rejected with model HTTP 422.")
else:
    raise AssertionError("The strict context window did not reject oversized state.")

### Bounded multi-question scaling

Run 1, 8, 32, and 64 independent question IDs over the same short state. These
are repeatability and hosting checks, not a decision-quality benchmark.
The default exact runtime batches questions and repeats their state; it may
split larger requests into several forward batches. Shared-context reuse is
disabled by default and is not enabled in this example.

Measure three requests per question count. Server latency excludes SDK/network
time; round-trip latency includes it. GPU peak allocation covers the most
recent inference and includes resident model allocations. Worker RSS is the
lifetime high-water mark, including startup and every preceding request.
Neither metric includes every process on the hosting instance.
These results apply only to this short input, runtime, and instance.

In [ ]:
scaling_results = []
for count in [1, 8, 32, 64]:
    payload = {
        "state": "The user explicitly requested the current weather in Paris.",
        "questions": {
            f"city_{index}": {
                "type": "noul",
                "instructions": "Did the user explicitly provide a city?",
            }
            for index in range(count)
        },
    }
    server_times, round_trip_times = [], []
    baseline_answers = None
    for _ in range(3):
        started = time.perf_counter()
        response = invoke(payload)
        round_trip_times.append((time.perf_counter() - started) * 1000)
        assert set(response["answers"]) == set(payload["questions"])
        for answer in response["answers"].values():
            assert answer["type"] == "noul" and math.isfinite(answer["noul"])
            assert 0 <= answer["noul"] <= 1
        if baseline_answers is None:
            baseline_answers = response["answers"]
        else:
            assert response["answers"] == baseline_answers
        server_times.append(response["latency_ms"])
    memory = invoke({"operation": "diagnostics"})
    scaling_results.append({
        "questions": count,
        "server_median_ms": round(statistics.median(server_times), 2),
        "round_trip_median_ms": round(statistics.median(round_trip_times), 2),
        "gpu_peak_allocated_bytes": memory["gpu_peak_allocated_bytes"],
        "gpu_peak_scope": memory["gpu_peak_scope"],
        "worker_peak_rss_bytes": memory["worker_peak_rss_bytes"],
        "worker_peak_rss_scope": "worker_lifetime",
    })
print(json.dumps(scaling_results, indent=2))

## 6. Cleanup

Delete this example's endpoint and wait before deleting its configuration and
model. Remove every S3 version and delete marker for this example's unique
artifact key, including repeated uploads. Preserve the shared bucket and role.
Attempt independent cleanup even if another operation fails; report failures.

Late endpoint log delivery can recreate the CloudWatch log group after deletion.
Rerun cleanup after delivery settles if the group reappears.

This cell keeps the serving image, its Amazon ECR repository (`decision-model-serving`),
and the AWS CodeBuild project (`decision-model-serving-build`), so later runs can reuse
them: set `SAGEMAKER_INFERENCE_IMAGE_URI` to the image printed above. The image is about
10 GB and ECR bills for its storage. To remove them:

```bash
aws ecr delete-repository --repository-name decision-model-serving --force
aws codebuild delete-project --name decision-model-serving-build
```

In [ ]:
def ignore_missing(operation, **arguments):
    try:
        return operation(**arguments)
    except ClientError as error:
        message = error.response["Error"].get("Message", "")
        error_code = error.response["Error"]["Code"]
        if error_code == "ResourceNotFoundException" or (
            error_code == "ValidationException"
            and ("Could not find" in message or "does not exist" in message)
        ):
            return None
        raise

def delete_endpoint():
    if ignore_missing(sm.describe_endpoint, EndpointName=name) is not None:
        sm.delete_endpoint(EndpointName=name)
        sm.get_waiter("endpoint_deleted").wait(
            EndpointName=name, WaiterConfig={"Delay": 15, "MaxAttempts": 120}
        )

def artifact_versions():
    return [
        {"Key": item["Key"], "VersionId": item["VersionId"]}
        for page in s3.get_paginator("list_object_versions").paginate(
            Bucket=bucket, Prefix=artifact_key
        )
        for group in ("Versions", "DeleteMarkers")
        for item in page.get(group, [])
        if item["Key"] == artifact_key
    ]

def delete_artifact():
    for item in artifact_versions():
        s3.delete_object(Bucket=bucket, **item)
    if artifact_versions():
        raise RuntimeError("S3 artifact versions remain after cleanup.")

cleanup_errors = []
def attempt_cleanup(label, operation, *positional, **arguments):
    try:
        operation(*positional, **arguments)
        return True
    except Exception as error:
        cleanup_errors.append(f"{label}: {error}")
        return False

try:
    if attempt_cleanup("endpoint", delete_endpoint):
        attempt_cleanup(
            "endpoint configuration", ignore_missing,
            sm.delete_endpoint_config, EndpointConfigName=name,
        )
        attempt_cleanup("model", ignore_missing, sm.delete_model, ModelName=name)
    attempt_cleanup("S3 artifact", delete_artifact)
    attempt_cleanup(
        "endpoint logs", ignore_missing,
        logs.delete_log_group, logGroupName=f"/aws/sagemaker/Endpoints/{name}",
    )
finally:
    attempt_cleanup("local archive", archive_path.unlink, missing_ok=True)

if cleanup_errors:
    raise RuntimeError("Cleanup incomplete: " + "; ".join(cleanup_errors))
assert ignore_missing(sm.describe_endpoint, EndpointName=name) is None
assert ignore_missing(sm.describe_endpoint_config, EndpointConfigName=name) is None
assert ignore_missing(sm.describe_model, ModelName=name) is None
print("Endpoint, configuration, model, all artifact versions, and endpoint logs cleaned up.")